# Petrol Price Forecasting — R2 pipeline (PLOS ONE, PONE-D-26-38172)

This notebook supersedes `Petrol_Price_predictions_R1.ipynb`. It keeps the leakage-free protocols of R1
(chronological 70/15/15 hold-out, rolling-origin 2019–2026, identical tuning budget, 5 seeds, DM tests, event windows,
ablation, SHAP with resampled backgrounds) and makes the following **pre-specified** changes, each motivated by a
diagnosis of the R1 results rather than by the R1 scores themselves:

| Change | Why |
|---|---|
| **Crude oil look-ahead removed** (`CRUDE_LAG_FIX`) | Before Aug 2023 the crude series holds month-end WTI settlements copied to every week of that month (up to 4 weeks of look-ahead). Each week now uses the previous month's settlement, which is known in that week. |
| **Level-invariant inputs and target** (`TARGET_MODE = 'diff'`) | In R1 the Pakistani test prices (248–381 PKR) lay far above the training maximum (161 PKR), so min–max-scaled networks had to extrapolate. All models now see weekly *changes* of the four variables (standardised on training weeks) and forecast the next weekly price *change*; the price forecast is last price + forecast change. ADF tests show the first differences are stationary. |
| **Persistence-initialised output** | The output layer of every network starts at zero, so an untrained network reproduces naive persistence and only moves away from it if the training data justify it. |
| **Autoregressive highway** (LSTNet-style) | A linear skip from the most recent input changes to the output is added to every network, so the nonlinear part only has to model what a linear AR model cannot. |
| **Adaptive pooling** | Max pooling is applied only when the window has ≥ 4 weeks (pooling a 1–3-week window discards the most recent week). |
| **Huber loss in the search space** | Administered prices change in rare, large steps; the random search can choose MSE or Huber loss. |
| **SVR on the same inputs** | SVR uses exactly the same differenced windows and target as the networks. |
| **Level-target sensitivity run** | The five networks are re-run with the R1 level target (same architecture, same settings) so the paper can report both representations. |
| **Bug fixes** | Granger tests (statsmodels ≥ 0.15 removed `verbose`), Grad-CAM for one-week windows. |

All five neural architectures receive every change identically, and the tuning budget is unchanged, so the comparison
remains fair. **Report whatever this run produces**, including if persistence still wins.

**Running on a laptop (VS Code / Jupyter):** put the two raw CSV files next to this notebook (or set
`PETROL_DATA_DIR`), then *Run All*. Everything is written to `./R2_outputs/` and every trained network is checkpointed,
so if the run stops you can simply *Run All* again and it resumes. Set environment variable `PETROL_FAST=1` for a
5-minute smoke test (results NOT for the paper).


In [ ]:
# Install once (uncomment if needed):
# %pip install -q tensorflow shap statsmodels openpyxl scikit-learn


## 0. Configuration

In [ ]:
import os, gc, json, time, random, itertools, platform, warnings, pickle, hashlib, shutil
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in globals() else False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/'
    DRIVE_ROOT = '/content/drive/MyDrive/Petrol_R2'      # ALL outputs and checkpoints are written here
else:
    DATA_DIR = os.environ.get('PETROL_DATA_DIR', './')
    DRIVE_ROOT = os.environ.get('PETROL_OUT_ROOT', '.')

PK_FILE = 'Petrol Price dataset updated_2026-1.csv'
SA_FILE = 'Saudi_PetrolFinalComplete.csv'
FAST_MODE = bool(int(os.environ.get('PETROL_FAST', '0')))   # True = smoke test only
# Smoke-test results go to a separate folder so they can never be mixed with the full (paper) run.
OUT = os.environ.get('PETROL_OUT_DIR', os.path.join(DRIVE_ROOT, 'R2_outputs_FAST' if FAST_MODE else 'R2_outputs'))

FEATURES = ['Exchange_Rate', 'Uncertainty_Index', 'Global_Oil_Price', 'Domestic_Price']
TARGET = 'Domestic_Price'
DL_MODELS = ['LSTM', 'BiLSTM', 'CNNLSTM', 'CNNLSTMAttention', 'CNNGRU']
MODEL_LABEL = {'LSTM': 'LSTM', 'BiLSTM': 'Bi-LSTM', 'CNNLSTM': 'CNN-LSTM', 'CNNLSTMAttention': 'CNN-LSTM-Att',
               'CNNGRU': 'CNN-GRU', 'CNNOnly': 'CNN only', 'Naive': 'Naive (persistence)',
               'SeasonalNaive': 'Seasonal naive (52 wk)', 'RWDrift': 'Random walk + drift',
               'ARIMA': 'ARIMA', 'ARIMAX': 'ARIMAX', 'SVR': 'SVR'}
PROPOSED = 'CNNLSTM'
HORIZON = 1                       # forecast horizon in WEEKS (one-week-ahead)
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15 # chronological hold-out; test = last 15 %
STEP_SIZES = [1, 2, 3, 4, 5, 8, 10, 12, 15]   # look-back windows in WEEKS
SEEDS = [0, 1, 2, 3, 4]
N_TRIALS = 20                     # random-search budget per DL model (identical for all models)
EPOCHS, PATIENCE = 200, 30

# Rolling-origin (expanding-window) evaluation
ROLL_START = '2019-01-01'         # first out-of-sample week
ROLL_BLOCK = 52                   # weeks per test block (one fold ≈ one year)
ROLL_SEEDS = [0, 1, 2]
ROLL_L_CANDIDATES = [1, 2, 4, 8, 12] # look-back re-selected inside every fold on that fold's validation block
ROLL_VAL_FRAC = 0.15

# Event windows for segmented (crisis) evaluation — AUTHORS: confirm the dates used in the manuscript.
EVENT_WINDOWS = {
    'Pre-COVID (tranquil)':          ('2019-01-01', '2020-02-29'),
    'COVID-19 shock':                ('2020-03-01', '2021-06-30'),
    'Post-COVID recovery':           ('2021-07-01', '2022-02-23'),
    'Russia-Ukraine war':            ('2022-02-24', '2022-12-31'),
    'Tranquil 2023-25':              ('2023-01-01', '2025-05-31'),
    'Iran-US/Israel escalation':     ('2025-06-01', '2026-04-30'),
}

# SHAP
SHAP_BG_RESAMPLES = 5
SHAP_BG_SIZE = 100
SHAP_BG_KMEANS = 25
SHAP_NSAMPLES = 512
SHAP_BOOT = 1000

ABLATION_SEEDS = [0, 1, 2, 3, 4]

# ---- R2 changes (see header) ----
TARGET_MODE = 'diff'              # 'diff': weekly changes in, next weekly change out (level-invariant); 'level': R1 behaviour
CRUDE_LAG_FIX = True              # use the previous month's WTI settlement for weeks in the monthly-data period
CRUDE_WEEKLY_FROM = '2023-08-01'  # from this date the crude series is genuinely weekly
AR_HIGHWAY_LAGS = 4               # linear autoregressive skip over the most recent (up to) 4 weeks of input changes
RUN_LEVEL_COMPARISON = True       # re-run the five networks with the R1 level target for the sensitivity table
ABLATION_PROTOCOL = {'Pakistan': 'holdout', 'Saudi Arabia': 'rolling'}  # Saudi hold-out test is a flat line

if FAST_MODE:
    STEP_SIZES = [2, 4]; SEEDS = [0]; N_TRIALS = 2; EPOCHS, PATIENCE = 3, 2
    ROLL_SEEDS = [0]; ROLL_L_CANDIDATES = [1, 2]; SHAP_BG_RESAMPLES = 2; SHAP_NSAMPLES = 40
    SHAP_BG_SIZE = 20; SHAP_BG_KMEANS = 5; SHAP_BOOT = 50; ABLATION_SEEDS = [0]

for d in ['', 'cache', 'cache/fits', 'cache/baselines', 'cache/shap', 'tables', 'figures', 'data', 'models']:
    os.makedirs(os.path.join(OUT, d), exist_ok=True)
print('Outputs and checkpoints ->', OUT)


### Resume check
Copies any results that an older version of this notebook wrote to the Colab local disk (`/content/R1_outputs`) into Drive. This only helps if the Colab machine survived the disconnect. It then lists what is already finished.


In [ ]:
LOCAL_OLD = '/content/R2_outputs'
if IN_COLAB and not FAST_MODE and os.path.isdir(LOCAL_OLD) and os.path.abspath(LOCAL_OLD) != os.path.abspath(OUT):
    n = 0
    for root, _, files in os.walk(LOCAL_OLD):
        for fn in files:
            s_ = os.path.join(root, fn); d_ = os.path.join(OUT, os.path.relpath(s_, LOCAL_OLD))
            if not os.path.exists(d_) and not fn.endswith('.tmp'):
                os.makedirs(os.path.dirname(d_), exist_ok=True); shutil.copy2(s_, d_); n += 1
    print(f'Recovered {n} files from {LOCAL_OLD} into {OUT}')

done = sorted(f[:-4] for f in os.listdir(f'{OUT}/cache') if f.endswith('.pkl'))
print('Finished stages on Drive:', done if done else 'none yet (fresh start)')
print('Checkpointed networks:', len(os.listdir(f'{OUT}/cache/fits')),
      '| baselines:', len(os.listdir(f'{OUT}/cache/baselines')), '| SHAP resamples:', len(os.listdir(f'{OUT}/cache/shap')))


## 1. Imports, determinism and software environment (Reviewer 4)

In [ ]:
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg') if not IN_COLAB else None
import matplotlib.pyplot as plt
import scipy, sklearn, statsmodels
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVR
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.tsa.statespace.sarimax import SARIMAX
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import shap

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

ENV = {
    'python': platform.python_version(), 'platform': platform.platform(),
    'tensorflow': tf.__version__, 'keras': tf.keras.__version__ if hasattr(tf.keras, '__version__') else 'bundled',
    'numpy': np.__version__, 'pandas': pd.__version__, 'scikit-learn': sklearn.__version__,
    'statsmodels': statsmodels.__version__, 'scipy': scipy.__version__, 'shap': shap.__version__,
    'matplotlib': matplotlib.__version__,
    'gpu': [d.name for d in tf.config.list_physical_devices('GPU')], 'fast_mode': FAST_MODE,
    'target_mode': TARGET_MODE, 'crude_lag_fix': CRUDE_LAG_FIX,
}
json.dump(ENV, open(f'{OUT}/environment.json', 'w'), indent=2)
print(json.dumps(ENV, indent=2))

KEY = {}   # placeholder-ID -> value  (written to R1_key_numbers.csv at the end)
def key(name, value, fmt=None):
    KEY[name] = (fmt.format(value) if (fmt and value is not None and not (isinstance(value, float) and np.isnan(value))) else value)
    return value

def set_seed(s):
    random.seed(s); np.random.seed(s); tf.keras.utils.set_random_seed(s)

_MISS = object()

def _atomic_pickle(obj, path):
    """Write to a temporary file first, then rename: a disconnect during the write never leaves a broken checkpoint."""
    tmp = path + '.tmp'
    with open(tmp, 'wb') as fh:
        pickle.dump(obj, fh)
    os.replace(tmp, path)

def _load_pickle(path):
    try:
        with open(path, 'rb') as fh:
            return pickle.load(fh)
    except Exception as e:                       # truncated / corrupted file -> delete and recompute
        print('  [cache] unreadable checkpoint, recomputing:', os.path.basename(path), '|', e)
        try: os.remove(path)
        except OSError: pass
        return _MISS

FIT_STATS = {'loaded': 0, 'trained': 0}

def cache(name, fn):
    """Run fn() once and pickle the result on Google Drive so an interrupted Colab run can resume.
    Inside every stage each network / baseline is cached as well (see fit_eval), so an interrupted
    stage resumes from the last finished network rather than from the beginning of the stage."""
    p = f'{OUT}/cache/{name}.pkl'
    if os.path.exists(p):
        r = _load_pickle(p)
        if r is not _MISS:
            if '/' not in name: print(f'[cache] {name}: loaded from Drive')
            return r
    t0 = time.time(); before = dict(FIT_STATS); r = fn(); _atomic_pickle(r, p)
    if '/' not in name:
        print(f"[cache] {name}: finished and saved to Drive in {(time.time() - t0) / 60:.1f} min "
              f"(networks loaded from checkpoints: {FIT_STATS['loaded'] - before['loaded']}, "
              f"newly trained: {FIT_STATS['trained'] - before['trained']})")
    return r


## 2. Data loading, description, VIF, stationarity and Granger tests (Editor-2, R1-4, R1-12, R1-13, R3-10, R4)

In [ ]:
RENAME = {
    'Pakistan': {'dollor price': 'Exchange_Rate', 'uncertainty Index': 'Uncertainty_Index',
                 'Global Price Crude': 'Global_Oil_Price', 'Pakistan': 'Domestic_Price'},
    'Saudi Arabia': {'Doller to Riyal Price': 'Exchange_Rate', 'Uncertinity Index Sudi Arebia': 'Uncertainty_Index',
                     'Global Price Crude': 'Global_Oil_Price', 'Saudi Arabia Riyal': 'Domestic_Price'},
}
UNITS = {'Pakistan': 'PKR/litre', 'Saudi Arabia': 'SAR/litre'}

def load(country, fname):
    df = pd.read_csv(os.path.join(DATA_DIR, fname))
    df.columns = [c.strip() for c in df.columns]
    df = df.rename(columns=RENAME[country])
    df['Date'] = pd.to_datetime(df['Period End'], format='%d-%b-%y')
    df = df.sort_values('Date').reset_index(drop=True)[['Date'] + FEATURES]
    if CRUDE_LAG_FIX:
        df = fix_crude(df)
    return df

def fix_crude(df):
    """Before CRUDE_WEEKLY_FROM the crude series holds the month-END settlement copied to every week of that month,
    i.e. up to ~4 weeks of look-ahead. Replace it by the PREVIOUS month's settlement, which is known in every week of
    the month. Weekly values from CRUDE_WEEKLY_FROM onwards are kept unchanged."""
    df = df.copy(); cut = pd.Timestamp(CRUDE_WEEKLY_FROM)
    mon = df.Date.dt.to_period('M')
    month_val = df[df.Date < cut].groupby(mon[df.Date < cut]).Global_Oil_Price.first()   # one value per month
    prev = month_val.shift(1)
    prev.iloc[0] = month_val.iloc[0]                       # first month (Jan 2013): no earlier value in the data
    m = df.Date < cut
    df.loc[m, 'Global_Oil_Price'] = mon[m].map(prev).values
    # first weekly weeks: keep weekly values (known at the end of each week)
    return df

DATA = {'Pakistan': load('Pakistan', PK_FILE), 'Saudi Arabia': load('Saudi Arabia', SA_FILE)}
CC = {'Pakistan': 'PK', 'Saudi Arabia': 'SA'}

desc_rows = []
for c, df in DATA.items():
    df.to_csv(f'{OUT}/data/{CC[c]}_weekly_dataset.csv', index=False)   # file deposited in the public repository
    y = df[TARGET]
    for f in FEATURES:
        s = df[f]
        runs = s.groupby((s != s.shift()).cumsum()).size()
        desc_rows.append({'Country': c, 'Variable': f, 'N': len(s), 'Missing': int(s.isna().sum()),
                          'Mean': s.mean(), 'SD': s.std(), 'Min': s.min(), 'Max': s.max(),
                          'Share of weeks with a change (%)': 100 * (s.diff().iloc[1:] != 0).mean(),
                          'Median run length (weeks)': runs.median()})
        if f == 'Uncertainty_Index': key(f'{CC[c]}_WUI_RUN', runs.mean(), '{:.1f}')
    key(f'{CC[c]}_N_OBS', len(df)); key(f'{CC[c]}_START', str(df.Date.min().date())); key(f'{CC[c]}_END', str(df.Date.max().date()))
    key(f'{CC[c]}_N_PRICE_CHANGES', int((y.diff().iloc[1:] != 0).sum()))
    key(f'{CC[c]}_N_MISSING', int(df[FEATURES].isna().sum().sum()))
DESC = pd.DataFrame(desc_rows); print(DESC.round(3).to_string())
last_change_sa = DATA['Saudi Arabia'].loc[DATA['Saudi Arabia'][TARGET].diff() != 0, 'Date'].max()
key('SA_LAST_PRICE_CHANGE', str(last_change_sa.date()))

# --- VIF on the four raw inputs (levels) and on first differences -------------------------------------------
def vif_table(X):
    Xc = (X - X.mean()) / X.std()
    Xc = Xc.assign(const=1.0)
    return {col: variance_inflation_factor(Xc.values, i) for i, col in enumerate(Xc.columns) if col != 'const'}

vif_rows, adf_rows, gc_rows = [], [], []
for c, df in DATA.items():
    lv = vif_table(df[FEATURES]); dv = vif_table(df[FEATURES].diff().dropna())
    for f in FEATURES:
        vif_rows.append({'Country': c, 'Variable': f, 'VIF (levels)': lv[f], 'VIF (first differences)': dv[f]})
        key(f'{CC[c]}_VIF_LEV_{f}', lv[f], '{:.2f}'); key(f'{CC[c]}_VIF_DIF_{f}', dv[f], '{:.2f}')
    for f in FEATURES:
        for kind, s in [('level', df[f]), ('diff', df[f].diff().dropna())]:
            try:
                p = adfuller(s, autolag='AIC')[1]
            except Exception:
                p = np.nan
            adf_rows.append({'Country': c, 'Variable': f, 'Series': kind, 'ADF p-value': p})
    # Granger causality: exogenous -> domestic price, first differences, lags 1..4 (predictive, NOT structural causality)
    d = df[FEATURES].diff().dropna()
    for f in ['Exchange_Rate', 'Uncertainty_Index', 'Global_Oil_Price']:
        try:
            try:
                res = grangercausalitytests(d[[TARGET, f]], maxlag=4, verbose=False)
            except TypeError:                     # statsmodels >= 0.15 removed the 'verbose' argument
                import contextlib, io
                with contextlib.redirect_stdout(io.StringIO()):
                    res = grangercausalitytests(d[[TARGET, f]], maxlag=4)
            for lag, r in res.items():
                gc_rows.append({'Country': c, 'Cause': f, 'Lag': lag, 'F': r[0]['ssr_ftest'][0], 'p': r[0]['ssr_ftest'][1]})
        except Exception as e:
            print('Granger test failed:', c, f, e)
            gc_rows.append({'Country': c, 'Cause': f, 'Lag': np.nan, 'F': np.nan, 'p': np.nan})
VIF = pd.DataFrame(vif_rows); ADF = pd.DataFrame(adf_rows); GC = pd.DataFrame(gc_rows)
print(VIF.round(2).to_string()); print(GC.round(4).to_string())
for c in DATA:
    key(f'{CC[c]}_VIF_MAX_LEV', VIF[VIF.Country == c]['VIF (levels)'].max(), '{:.2f}')
    key(f'{CC[c]}_VIF_MAX_DIF', VIF[VIF.Country == c]['VIF (first differences)'].max(), '{:.2f}')

## 3. Windowing, splits, scaling, metrics and the Diebold–Mariano test

In [ ]:
def make_windows(arr, L, h=HORIZON, tcol=FEATURES.index(TARGET)):
    """Inputs: weeks t-L+1..t (all 4 variables, observed at the end of week t). Target: price in week t+h."""
    X, y, t_idx, last = [], [], [], []
    for t in range(L - 1, len(arr) - h):
        X.append(arr[t - L + 1:t + 1]); y.append(arr[t + h, tcol]); t_idx.append(t + h); last.append(arr[t, tcol])
    return np.asarray(X, float), np.asarray(y, float), np.asarray(t_idx), np.asarray(last, float)

def holdout_bounds(n):
    return int(n * TRAIN_FRAC), int(n * (TRAIN_FRAC + VAL_FRAC))

def make_windows_diff(arr, L, h=HORIZON, tcol=FEATURES.index(TARGET)):
    """Level-invariant windows. Inputs: weekly CHANGES of all variables in weeks t-L+1..t (each change uses the
    observations of weeks s-1 and s, both known at the end of week t). Target: price change y[t+h] - y[t]."""
    D = np.vstack([np.full((1, arr.shape[1]), np.nan), np.diff(arr, axis=0)])
    X, dy, t_idx, last, y = [], [], [], [], []
    for t in range(L, len(arr) - h):
        X.append(D[t - L + 1:t + 1]); dy.append(arr[t + h, tcol] - arr[t, tcol]); t_idx.append(t + h)
        last.append(arr[t, tcol]); y.append(arr[t + h, tcol])
    return (np.asarray(X, float), np.asarray(dy, float), np.asarray(t_idx), np.asarray(last, float), np.asarray(y, float), D)

def prepare(df, L, train_end, val_end, test_end=None, feats=FEATURES, mode=None):
    """Chronological split on the TARGET week index. Scalers are fitted on training weeks only.
    mode='diff' (default, R2): inputs = standardised weekly changes, target = standardised next price change,
    forecast = last price + de-scaled forecast change. mode='level': the R1 min-max level representation."""
    mode = mode or TARGET_MODE
    arr = df[feats].values.astype(float)
    tcol = feats.index(TARGET)
    test_end = len(df) if test_end is None else test_end
    if mode == 'level':
        X, y, ti, last = make_windows(arr, L, tcol=tcol); tgt = y
        sx = MinMaxScaler().fit(arr[:train_end]); sy = MinMaxScaler().fit(arr[:train_end, [tcol]])
    else:
        X, tgt, ti, last, y, D = make_windows_diff(arr, L, tcol=tcol)
        sx = StandardScaler().fit(D[1:train_end]); sy = StandardScaler().fit(tgt[ti < train_end].reshape(-1, 1))
    m_tr = ti < train_end; m_va = (ti >= train_end) & (ti < val_end); m_te = (ti >= val_end) & (ti < test_end)
    sc = lambda A: sx.transform(A.reshape(-1, A.shape[-1])).reshape(A.shape)
    out = {}
    for nm, m in [('tr', m_tr), ('va', m_va), ('te', m_te)]:
        out[nm] = dict(X=sc(X[m]), y=y[m], ys=sy.transform(tgt[m].reshape(-1, 1)).ravel(), t=ti[m], last=last[m])
    out['sy'] = sy; out['L'] = L; out['F'] = len(feats); out['y_train_raw'] = arr[:train_end, tcol]
    out['dates'] = df['Date'].values; out['mode'] = mode
    return out

def to_price(P, part, z):
    """Convert scaled network/SVR output for block `part` into a price forecast (PKR or SAR per litre)."""
    d = P['sy'].inverse_transform(np.asarray(z, float).reshape(-1, 1)).ravel()
    return P[part]['last'] + d if P['mode'] == 'diff' else d

def y_unit(P):
    """Price units per unit of the scaled target (used to express SHAP values in PKR or SAR)."""
    return float(P['sy'].scale_[0]) if P['mode'] == 'diff' else float(P['sy'].data_max_[0] - P['sy'].data_min_[0])

def scaler_state(P):
    sy = P['sy']
    return [P['mode']] + [np.asarray(getattr(sy, a)) for a in ('mean_', 'scale_', 'data_min_', 'data_max_') if hasattr(sy, a) and getattr(sy, a) is not None]

def metrics(y, yhat, y_naive=None, y_train=None, n_params=None):
    y, yhat = np.asarray(y, float), np.asarray(yhat, float)
    e = y - yhat; n = len(y)
    mae = np.mean(np.abs(e)); mse = np.mean(e ** 2); rmse = np.sqrt(mse)
    sst = np.sum((y - y.mean()) ** 2)
    r2 = 1 - np.sum(e ** 2) / sst if sst > 1e-12 else np.nan          # undefined for a constant test target
    adj = (1 - (1 - r2) * (n - 1) / (n - n_params - 1)) if (n_params and n - n_params - 1 > 0 and not np.isnan(r2)) else np.nan
    mape = 100 * np.mean(np.abs(e) / np.abs(y))
    rrmse = 100 * rmse / np.mean(y)                                     # relative RMSE (% of mean price)
    out = dict(MAE=mae, MSE=mse, RMSE=rmse, rRMSE=rrmse, MAPE=mape, R2=r2, R2_adj=adj)
    if y_naive is not None:
        rn = np.sqrt(np.mean((y - y_naive) ** 2))
        out['U2'] = rmse / rn if rn > 1e-12 else np.nan                 # Theil's U2 (<1 = beats persistence)
    if y_train is not None:
        scale = np.mean(np.abs(np.diff(y_train)))
        out['MASE'] = mae / scale if scale > 1e-12 else np.nan
    return out

def dm_test(e1, e2, h=HORIZON, power=2):
    """Diebold-Mariano test with the Harvey-Leybourne-Newbold small-sample correction.
    H0: equal accuracy. Negative statistic => model 1 more accurate."""
    e1, e2 = np.asarray(e1), np.asarray(e2)
    d = np.abs(e1) ** power - np.abs(e2) ** power
    n = len(d); dbar = d.mean()
    gam = [np.mean((d[k:] - dbar) * (d[:n - k] - dbar)) for k in range(h)]
    v = (gam[0] + 2 * sum(gam[1:])) / n
    if not np.isfinite(v) or v <= 1e-18:
        return np.nan, np.nan
    hln = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    s = hln * dbar / np.sqrt(v)
    return s, 2 * (1 - stats.t.cdf(abs(s), df=n - 1))

def ci95(x):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) < 2: return (np.nan, np.nan)
    hw = stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))
    return (x.mean() - hw, x.mean() + hw)

## 4. Model builders (all five DL architectures share one parametrised builder)

In [ ]:
_COMMON = dict(loss='mse', ar=1)   # R2: every network has the AR highway and starts at persistence
DEFAULT_HP = {   # configuration of the submitted manuscript (v5 notebook) + R2 common settings
    'LSTM':             dict(n_rec=2, units=64, dense=16, dropout=0.1, lr=1e-3, batch=32, **_COMMON),
    'BiLSTM':           dict(n_rec=2, units=64, dense=16, dropout=0.1, lr=1e-3, batch=32, **_COMMON),
    'CNNLSTM':          dict(n_conv=1, filters=64, kernel=2, n_rec=1, units=50, dense=16, dropout=0.1, lr=1e-3, batch=32, **_COMMON),
    'CNNLSTMAttention': dict(n_conv=1, filters=64, kernel=2, n_rec=1, units=64, dense=32, dropout=0.1, lr=1e-3, batch=32, heads=4, **_COMMON),
    'CNNGRU':           dict(n_conv=1, filters=64, kernel=2, n_rec=1, units=50, dense=16, dropout=0.1, lr=1e-3, batch=32, **_COMMON),
    'CNNOnly':          dict(n_conv=1, filters=64, kernel=2, n_rec=0, units=0, dense=16, dropout=0.1, lr=1e-3, batch=32, **_COMMON),
}
SEARCH_SPACE = dict(filters=[32, 64, 128], kernel=[2, 3], n_conv=[1, 2], units=[32, 50, 64, 128], n_rec=[1, 2],
                    dense=[16, 32], dropout=[0.0, 0.1, 0.2, 0.3], lr=[1e-3, 5e-4], batch=[16, 32], loss=['mse', 'huber'])
HP_FLOAT, HP_STR = ('dropout', 'lr'), ('loss',)
def cast_hp(k, v):
    return float(v) if k in HP_FLOAT else (str(v) if k in HP_STR else int(v))
CONV_MODELS = {'CNNLSTM', 'CNNGRU', 'CNNLSTMAttention', 'CNNOnly'}

def build(name, L, F, hp):
    inp = layers.Input((L, F)); x = inp
    if name in CONV_MODELS:
        for i in range(hp['n_conv']):
            x = layers.Conv1D(hp['filters'], hp['kernel'], padding='same', activation='relu', name=f'conv{i}')(x)
        if L >= 4:   # R2: pooling a 1-3 week window would merge (and dilute) the most recent week
            x = layers.MaxPooling1D(pool_size=2, padding='same')(x)
    if name == 'CNNOnly':
        x = layers.Flatten()(x)
        if hp['dropout'] > 0: x = layers.Dropout(hp['dropout'])(x)
    elif name == 'CNNLSTMAttention':
        for i in range(hp['n_rec']):
            x = layers.LSTM(hp['units'], return_sequences=True)(x)
        a = layers.MultiHeadAttention(num_heads=hp.get('heads', 4), key_dim=max(4, hp['units'] // 4))(x, x)
        a = layers.Dropout(hp['dropout'])(a)
        x = layers.LayerNormalization()(layers.Add()([x, a]))
        x = layers.GlobalAveragePooling1D()(x)
    else:
        cell = layers.GRU if name == 'CNNGRU' else layers.LSTM
        for i in range(hp['n_rec']):
            last = i == hp['n_rec'] - 1
            u = hp['units'] if i == 0 else max(16, hp['units'] // 2)
            lyr = cell(u, return_sequences=not last)
            x = layers.Bidirectional(lyr)(x) if name == 'BiLSTM' else lyr(x)
            if hp['dropout'] > 0: x = layers.Dropout(hp['dropout'])(x)
    x = layers.Dense(hp['dense'], activation='relu')(x)
    if name == 'CNNLSTMAttention' and hp['dropout'] > 0: x = layers.Dropout(hp['dropout'])(x)
    # R2: zero-initialised head -> an untrained network forecasts "no change" (= naive persistence in 'diff' mode)
    out = layers.Dense(1, kernel_initializer='zeros', bias_initializer='zeros', name='head')(x)
    if hp.get('ar', 0):
        # R2: LSTNet-style autoregressive highway: linear skip from the most recent input changes to the output
        k = min(L, AR_HIGHWAY_LAGS)
        z = layers.Cropping1D(cropping=(L - k, 0), name='ar_window')(inp) if k < L else inp
        z = layers.Dense(1, kernel_initializer='zeros', bias_initializer='zeros', name='ar_highway')(layers.Flatten()(z))
        out = layers.Add(name='output')([out, z])
    return Model(inp, out)

def fit_dl(name, hp, P, seed, epochs=None):
    tf.keras.backend.clear_session(); gc.collect(); set_seed(seed)
    m = build(name, P['L'], P['F'], hp)
    loss = tf.keras.losses.Huber(delta=1.0) if hp.get('loss', 'mse') == 'huber' else 'mse'
    m.compile(optimizer=tf.keras.optimizers.Adam(hp['lr']), loss=loss)
    cb = [EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True),
          ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6)]
    # mini-batches are shuffled WITHIN the training block only; validation/test weeks are never seen in training
    h = m.fit(P['tr']['X'], P['tr']['ys'], validation_data=(P['va']['X'], P['va']['ys']),
              epochs=epochs or EPOCHS, batch_size=hp['batch'], verbose=0, callbacks=cb, shuffle=True)
    return m, h.history

def predict(m, P, part):
    return to_price(P, part, m.predict(P[part]['X'], verbose=0))

def val_rmse(m, P):
    return float(np.sqrt(np.mean((P['va']['y'] - predict(m, P, 'va')) ** 2)))

def _fingerprint(*parts):
    h = hashlib.sha1()
    for p in parts:
        if isinstance(p, np.ndarray):
            h.update(str(p.shape).encode()); h.update(np.ascontiguousarray(p, dtype=np.float64).tobytes())
        else:
            h.update(json.dumps(p, sort_keys=True, default=str).encode())
    return h.hexdigest()[:20]

def fit_eval(name, hp, P, seed, save_model=None):
    """Train ONE network, or load its result from Drive if it was already trained in an earlier session.
    The checkpoint key is a hash of the architecture, hyper-parameters, seed, epochs and the exact
    training/validation/test arrays, so a cached result is only reused for an identical run.
    Returns validation RMSE, de-scaled validation and test predictions, training history and parameter count."""
    fp = _fingerprint(name, hp, int(seed), EPOCHS, PATIENCE, P['tr']['X'], P['tr']['ys'], P['va']['X'],
                      P['va']['ys'], P['te']['X'], *scaler_state(P))
    path = f'{OUT}/cache/fits/{name}_{fp}.pkl'
    need_model = save_model is not None and not os.path.exists(save_model)
    if os.path.exists(path) and not need_model:
        r = _load_pickle(path)
        if r is not _MISS:
            FIT_STATS['loaded'] += 1; return r
    m, hist = fit_dl(name, hp, P, seed)
    pv, pt = predict(m, P, 'va'), predict(m, P, 'te')
    r = dict(val_rmse=float(np.sqrt(np.mean((P['va']['y'] - pv) ** 2))), pred={'va': pv, 'te': pt},
             hist={k: [float(v) for v in vals] for k, vals in hist.items()}, n_params=int(m.count_params()))
    if save_model is not None:
        tmp = save_model[:-len('.keras')] + '.tmp.keras'; m.save(tmp); os.replace(tmp, save_model)
    _atomic_pickle(r, path); FIT_STATS['trained'] += 1
    return r

def sample_hp(rng, name):
    hp = {k: v[int(rng.integers(len(v)))] for k, v in SEARCH_SPACE.items()}
    hp = {k: cast_hp(k, v) for k, v in hp.items()}
    hp['ar'] = 1
    if name == 'CNNLSTMAttention': hp['heads'] = 4
    return hp

## 5. Chronological hold-out split (replaces the shuffled 68/12/20 split)

In [ ]:
SPLITS = {}
for c, df in DATA.items():
    n = len(df); a, b = holdout_bounds(n)
    SPLITS[c] = (a, b)
    key(f'{CC[c]}_TRAIN_END', str(df.Date[a - 1].date())); key(f'{CC[c]}_VAL_END', str(df.Date[b - 1].date()))
    key(f'{CC[c]}_TEST_START', str(df.Date[b].date())); key(f'{CC[c]}_N_TEST', n - b)
    key(f'{CC[c]}_TEST_SD', df[TARGET].iloc[b:].std(), '{:.3f}')
    key(f'{CC[c]}_TEST_N_CHANGES', int((df[TARGET].iloc[b - 1:].diff().iloc[1:] != 0).sum()))
    print(c, 'train ends', df.Date[a - 1].date(), '| val ends', df.Date[b - 1].date(), '| test', df.Date[b].date(), '→', df.Date.iloc[-1].date())

## 6. Look-back (window) selection and identical random hyper-parameter search on the VALIDATION set
Stage A: every DL model × every look-back L (weeks) with its default configuration, 3 seeds → choose L* by mean validation RMSE.
Stage B: N_TRIALS random configurations (same budget for every model) at L* → choose configuration by validation RMSE.
For Saudi Arabia the 2022–24 validation block is a constant price (2.33 SAR) — selection there is degenerate, so the
Saudi models reuse the default configuration and pick L* by mean validation RMSE over the rolling-origin folds (§9).

In [ ]:
def stage_a(c):
    df = DATA[c]; a, b = SPLITS[c]; rows = []
    for name in DL_MODELS:
        for L in STEP_SIZES:
            P = prepare(df, L, a, b)
            for s in SEEDS[:3]:
                r = fit_eval(name, DEFAULT_HP[name], P, s)
                rows.append({'Country': c, 'Model': name, 'L': L, 'seed': s, 'val_RMSE': r['val_rmse']})
            print(c, name, L, np.mean([r['val_RMSE'] for r in rows if r['Model'] == name and r['L'] == L]))
    return pd.DataFrame(rows)

def stage_b(c, Lstar):
    df = DATA[c]; a, b = SPLITS[c]; rows = []
    rng = np.random.default_rng(2026)
    for name in DL_MODELS:
        P = prepare(df, Lstar[name], a, b)
        trials = [DEFAULT_HP[name]] + [sample_hp(rng, name) for _ in range(N_TRIALS - 1)]
        for k, hp in enumerate(trials):
            r = fit_eval(name, hp, P, 0)
            rows.append({'Country': c, 'Model': name, 'trial': k, 'L': Lstar[name], **hp,
                         'n_params': r['n_params'], 'epochs': len(r['hist']['loss']), 'val_RMSE': r['val_rmse']})
        print(c, name, 'best trial val RMSE', min(r['val_RMSE'] for r in rows if r['Model'] == name))
    return pd.DataFrame(rows)

LB = cache('stageA_PK', lambda: stage_a('Pakistan'))
LB_SUM = LB.groupby(['Country', 'Model', 'L']).val_RMSE.agg(['mean', 'std']).reset_index()
LSTAR = {'Pakistan': {m: int(LB_SUM[(LB_SUM.Model == m)].sort_values('mean').iloc[0].L) for m in DL_MODELS}}
HS = cache('stageB_PK', lambda: stage_b('Pakistan', LSTAR['Pakistan']))
BEST_HP = {'Pakistan': {}}
for m in DL_MODELS:
    r = HS[HS.Model == m].sort_values('val_RMSE').iloc[0]
    BEST_HP['Pakistan'][m] = {k: cast_hp(k, r[k]) for k in DEFAULT_HP[m] if k in r and not pd.isna(r[k])}
    if m == 'CNNLSTMAttention': BEST_HP['Pakistan'][m]['heads'] = 4
print('Pakistan L*:', LSTAR['Pakistan']); print(json.dumps(BEST_HP['Pakistan'], indent=1))

# Architecture sensitivity of the proposed CNN-LSTM (depth / width), extracted from the random search (R1-5)
SENS = HS[HS.Model == 'CNNLSTM'][['n_conv', 'filters', 'kernel', 'n_rec', 'units', 'dense', 'dropout', 'lr', 'batch', 'loss', 'n_params', 'val_RMSE']].sort_values('val_RMSE')
SENS_DEPTH = SENS.groupby(['n_conv', 'n_rec']).val_RMSE.agg(['min', 'median', 'count']).reset_index()
print(SENS_DEPTH)

## 7. Benchmarks on identical splits: naive, seasonal naive, random walk + drift, ARIMA, ARIMAX, tuned SVR (R1-7, R2-12, R3-2)

In [ ]:
def _baselines_holdout(c, L_svr, train_end, val_end, test_end=None):
    df = DATA[c]; y = df[TARGET].values; n = len(df); test_end = n if test_end is None else test_end
    idx = np.arange(val_end, test_end)                                   # target weeks of the test block
    out = {}
    out['Naive'] = y[idx - HORIZON]
    out['SeasonalNaive'] = y[idx - 52]
    drift = (y[train_end - 1] - y[0]) / (train_end - 1)
    out['RWDrift'] = y[idx - HORIZON] + HORIZON * drift
    # ARIMA / ARIMAX: order chosen by AIC on the training block; exogenous regressors lagged one week (known at t)
    exog = df[['Exchange_Rate', 'Uncertainty_Index', 'Global_Oil_Price']].shift(1).bfill().values
    cfg = {}
    for kind in ['ARIMA', 'ARIMAX']:
        best = (np.inf, None)
        for p, d, q in itertools.product(range(0, 4), range(0, 3), range(0, 4)):
            try:
                r = SARIMAX(y[:train_end], exog=exog[:train_end] if kind == 'ARIMAX' else None, order=(p, d, q),
                            enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
                if r.aic < best[0]: best = (r.aic, (p, d, q))
            except Exception:
                pass
            if FAST_MODE and best[1] is not None: break
        order = best[1]; cfg[kind] = order
        r = SARIMAX(y[:train_end], exog=exog[:train_end] if kind == 'ARIMAX' else None, order=order,
                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        # one-step-ahead forecasts: parameters fixed at the training estimate, state updated with the realised data
        full = r.apply(y[:test_end], exog=exog[:test_end] if kind == 'ARIMAX' else None)
        out[kind] = full.predict(start=val_end, end=test_end - 1)
    # SVR on the same flattened multivariate windows as the DL models; grid chosen on the validation block
    P = prepare(df, L_svr, train_end, val_end, test_end)
    flat = lambda part: P[part]['X'].reshape(len(P[part]['X']), -1)
    best = (np.inf, None)
    for C, g, eps in itertools.product([0.1, 1, 10, 100], ['scale', 0.01, 0.1, 1.0], [0.001, 0.01, 0.1]):
        mdl = SVR(C=C, gamma=g, epsilon=eps).fit(flat('tr'), P['tr']['ys'])
        v = np.sqrt(np.mean((P['va']['y'] - to_price(P, 'va', mdl.predict(flat('va')))) ** 2))
        if v < best[0]: best = (v, (C, g, eps))
        if FAST_MODE: break
    C, g, eps = best[1]; cfg['SVR'] = dict(C=C, gamma=g, epsilon=eps, L=L_svr)
    mdl = SVR(C=C, gamma=g, epsilon=eps).fit(np.vstack([flat('tr')]), P['tr']['ys'])
    out['SVR'] = to_price(P, 'te', mdl.predict(flat('te')))
    assert np.array_equal(P['te']['t'], idx)
    return out, cfg, y[idx], y[idx - HORIZON], idx

def baselines_holdout(c, L_svr, train_end, val_end, test_end=None):
    """ARIMA/ARIMAX order search + SVR grid, cached on Drive per split (they are re-used by every rolling fold)."""
    return cache(f'baselines/{CC[c]}_{TARGET_MODE}_L{L_svr}_{train_end}_{val_end}_{test_end}',
                 lambda: _baselines_holdout(c, L_svr, train_end, val_end, test_end))


## 8. Hold-out evaluation: tuned DL models × 5 seeds, benchmarks, 95% CIs, DM tests (R3-7, R3-12)

In [ ]:
def run_holdout(c, hp_map, L_map):
    df = DATA[c]; a, b = SPLITS[c]
    base, cfg, y_te, y_naive, idx = baselines_holdout(c, L_map[PROPOSED], a, b)
    preds = {k: v for k, v in base.items()}
    rows, hist_store, seed_preds = [], {}, {}
    ytr = df[TARGET].values[:a]
    for name in DL_MODELS:
        P = prepare(df, L_map[name], a, b)
        assert np.array_equal(P['te']['t'], idx)
        ps = []
        for s in SEEDS:
            mpath = f'{OUT}/models/{CC[c]}_{name}_holdout_seed{s}.keras' if s == SEEDS[0] else None
            r = fit_eval(name, hp_map[name], P, s, save_model=mpath)
            p = r['pred']['te']; ps.append(p)
            mt = metrics(y_te, p, y_naive, ytr, n_params=P['L'] * P['F'])
            rows.append({'Country': c, 'Model': name, 'seed': s, **mt})
            if s == SEEDS[0]:
                hist_store[name] = r['hist']
        seed_preds[name] = np.array(ps); preds[name] = np.mean(ps, axis=0)
    for k in base:
        rows.append({'Country': c, 'Model': k, 'seed': -1, **metrics(y_te, base[k], y_naive, ytr, n_params=None)})
    R = pd.DataFrame(rows)
    return dict(R=R, preds=preds, seed_preds=seed_preds, y=y_te, y_naive=y_naive, idx=idx, cfg=cfg, hist=hist_store)

def summarise(R, metric_cols=('RMSE', 'MAE', 'rRMSE', 'MAPE', 'MASE', 'U2', 'R2')):
    out = []
    for (c, m), g in R.groupby(['Country', 'Model'], sort=False):
        row = {'Country': c, 'Model': m, 'n_seeds': int((g.seed >= 0).sum())}
        for k in metric_cols:
            v = g[k].values.astype(float)
            row[k] = np.nanmean(v); row[k + '_sd'] = np.nanstd(v, ddof=1) if len(v) > 1 else np.nan
            lo, hi = ci95(v); row[k + '_lo'], row[k + '_hi'] = lo, hi
        out.append(row)
    return pd.DataFrame(out)

def dm_table(res, c, ref=PROPOSED):
    rows = []
    e = {k: res['y'] - v for k, v in res['preds'].items()}
    for k in e:
        if k == ref: continue
        s, p = dm_test(e[ref], e[k]); rows.append({'Country': c, 'Model': ref, 'vs': k, 'DM': s, 'p': p})
    for k in DL_MODELS:
        s, p = dm_test(e[k], e['Naive']); rows.append({'Country': c, 'Model': k, 'vs': 'Naive', 'DM': s, 'p': p})
    return pd.DataFrame(rows)

HO = {}
HO['Pakistan'] = cache('holdout_PK', lambda: run_holdout('Pakistan', BEST_HP['Pakistan'], LSTAR['Pakistan']))

## 9. Rolling-origin (expanding-window) evaluation, 2019–2026 (both countries; primary protocol for Saudi Arabia)

In [ ]:
def roll_folds(df):
    start = int(np.searchsorted(df.Date.values, np.datetime64(ROLL_START)))
    folds, s = [], start
    while s < len(df):
        e = min(s + ROLL_BLOCK, len(df)); folds.append((s, e)); s = e
    return folds

def run_rolling(c, hp_map, models=DL_MODELS, feats=FEATURES, tag='main'):
    df = DATA[c]; y = df[TARGET].values; rows, oos = [], []
    for k, (s, e) in enumerate(roll_folds(df)):
        tr_end = int(s - ROLL_VAL_FRAC * s)                      # validation block = 15 % of weeks preceding the test block
        fold_pred = {}
        for name in models:
            # look-back re-selected inside the fold on the fold's own validation block
            best = (np.inf, None)
            for L in ROLL_L_CANDIDATES:
                P = prepare(df, L, tr_end, s, e, feats=feats)
                v = fit_eval(name, hp_map[name], P, 0)['val_rmse']
                if v < best[0]: best = (v, L)
            P = prepare(df, best[1], tr_end, s, e, feats=feats)
            ps = []
            for sd in ROLL_SEEDS:
                ps.append(fit_eval(name, hp_map[name], P, sd)['pred']['te'])
            fold_pred[name] = np.mean(ps, axis=0); t_idx = P['te']['t']
            rows.append({'Country': c, 'fold': k, 'Model': name, 'L': best[1], 'seed_sd_RMSE':
                         np.std([np.sqrt(np.mean((y[t_idx] - p) ** 2)) for p in ps])})
            print(c, tag, 'fold', k, name, 'L', best[1])
        gc.collect()
        if tag == 'main':
            base, cfg, _, _, idx = baselines_holdout(c, rows[-len(models) + models.index(PROPOSED)]['L'], tr_end, s, e)
            assert np.array_equal(idx, t_idx)
            fold_pred.update(base)
        for name, p in fold_pred.items():
            for t, v in zip(t_idx, p):
                oos.append({'Country': c, 'fold': k, 'Model': name, 't': int(t), 'Date': df.Date[t], 'y': y[t], 'yhat': float(v),
                            'y_naive': y[t - HORIZON]})
    return pd.DataFrame(rows), pd.DataFrame(oos)

# Saudi Arabia: default (submitted) configuration — its hold-out validation block is constant, so no tuning there.
HP_ROLL = {'Pakistan': BEST_HP['Pakistan'], 'Saudi Arabia': {m: DEFAULT_HP[m] for m in DL_MODELS}}
ROLL = {c: cache(f'rolling_{CC[c]}', lambda c=c: run_rolling(c, HP_ROLL[c])) for c in DATA}

def oos_metrics(O, groupcol=None):
    rows = []
    keys = ['Country', 'Model'] + ([groupcol] if groupcol else [])
    for kk, g in O.groupby(keys, sort=False):
        ytr = None
        mt = metrics(g.y.values, g.yhat.values, g.y_naive.values, None)
        rows.append({**dict(zip(keys, kk)), 'n_weeks': len(g), 'n_price_changes': int((g.y.values != g.y_naive.values).sum()), **mt})
    return pd.DataFrame(rows)

ROLL_ALL = pd.concat([oos_metrics(ROLL[c][1]) for c in DATA]); print(ROLL_ALL.round(4).to_string())

# Saudi hold-out is still run for completeness (default configuration, L* = median rolling-fold choice)
_RF = ROLL['Saudi Arabia'][0]
LSTAR['Saudi Arabia'] = {m: int(_RF[_RF.Model == m].L.median()) for m in DL_MODELS}
HO['Saudi Arabia'] = cache('holdout_SA', lambda: run_holdout('Saudi Arabia', HP_ROLL['Saudi Arabia'], LSTAR['Saudi Arabia']))

## 10. Event-window (crisis) evaluation and price-change weeks (R1-11, R3-5)

In [ ]:
def tag_event(d):
    for k, (a, b) in EVENT_WINDOWS.items():
        if pd.Timestamp(a) <= d <= pd.Timestamp(b): return k
    return 'Other'

EV = []
for c in DATA:
    O = ROLL[c][1].copy(); O['Window'] = O.Date.apply(tag_event)
    O['Week type'] = np.where(O.y != O.y_naive, 'price-change week', 'no-change week')
    EV.append(oos_metrics(O, 'Window').assign(Split='event'))
    EV.append(oos_metrics(O, 'Week type').rename(columns={'Week type': 'Window'}).assign(Split='change'))
EVENTS = pd.concat(EV); print(EVENTS[['Country', 'Model', 'Window', 'n_weeks', 'n_price_changes', 'RMSE', 'rRMSE', 'U2']].round(3).to_string())

# DM tests on the pooled rolling-origin forecasts
DM_ROLL = []
for c in DATA:
    O = ROLL[c][1]; W = O.pivot_table(index='t', columns='Model', values='yhat'); yy = O.drop_duplicates('t').set_index('t').y.loc[W.index]
    for k in W.columns:
        if k != PROPOSED:
            s, p = dm_test(yy - W[PROPOSED], yy - W[k]); DM_ROLL.append({'Country': c, 'Model': PROPOSED, 'vs': k, 'DM': s, 'p': p})
    for k in DL_MODELS:
        s, p = dm_test(yy - W[k], yy - W['Naive']); DM_ROLL.append({'Country': c, 'Model': k, 'vs': 'Naive', 'DM': s, 'p': p})
DM_ROLL = pd.DataFrame(DM_ROLL)
DM_HO = pd.concat([dm_table(HO[c], c) for c in DATA])

## 11. Ablation study with repeated seeds (R3-7)

In [ ]:
ABL_VARIANTS = {   # name: (architecture, feature set, dropout override)
    'Proposed CNN-LSTM (full)':      ('CNNLSTM', FEATURES, None),
    'CNN-LSTM without AR highway':   ('CNNLSTM', FEATURES, 'no_ar'),
    'LSTM only (no convolution)':    ('LSTM', FEATURES, None),
    'CNN only (no recurrence)':      ('CNNOnly', FEATURES, None),
    'CNN-LSTM without dropout':      ('CNNLSTM', FEATURES, 0.0),
    'CNN-LSTM price only':           ('CNNLSTM', [TARGET], None),
    'CNN-LSTM without exchange rate':('CNNLSTM', ['Uncertainty_Index', 'Global_Oil_Price', TARGET], None),
    'CNN-LSTM without uncertainty':  ('CNNLSTM', ['Exchange_Rate', 'Global_Oil_Price', TARGET], None),
    'CNN-LSTM without crude price':  ('CNNLSTM', ['Exchange_Rate', 'Uncertainty_Index', TARGET], None),
}

def run_ablation(c):
    df = DATA[c]; y = df[TARGET].values; rows, preds = [], {}
    base_hp = HP_ROLL[c][PROPOSED] if c == 'Saudi Arabia' else BEST_HP[c][PROPOSED]
    for vname, (arch, feats, dout) in ABL_VARIANTS.items():
        hp = dict(base_hp) if arch == PROPOSED else dict(BEST_HP[c].get(arch, DEFAULT_HP[arch]) if c == 'Pakistan' else DEFAULT_HP[arch])
        if arch == 'CNNOnly': hp = {**DEFAULT_HP['CNNOnly'], **{k: base_hp[k] for k in ('n_conv', 'filters', 'kernel', 'dense', 'dropout', 'lr', 'batch')}}
        if dout == 'no_ar': hp['ar'] = 0
        elif dout is not None: hp['dropout'] = dout
        if ABLATION_PROTOCOL[c] == 'holdout':
            a, b = SPLITS[c]; P = prepare(df, LSTAR[c][PROPOSED], a, b, feats=feats); ps = []
            for s in ABLATION_SEEDS:
                p = fit_eval(arch, hp, P, s)['pred']['te']; ps.append(p)
                rows.append({'Country': c, 'Variant': vname, 'seed': s, **metrics(P['te']['y'], p, P['te']['last'])})
            preds[vname] = (P['te']['y'], np.mean(ps, axis=0))
        else:
            r, O = run_rolling(c, {arch: hp}, models=[arch], feats=feats, tag='ablation')
            O = O[O.Model == arch]
            rows.append({'Country': c, 'Variant': vname, 'seed': -1, **metrics(O.y.values, O.yhat.values, O.y_naive.values)})
            preds[vname] = (O.y.values, O.yhat.values)
    R = pd.DataFrame(rows)
    full_y, full_p = preds['Proposed CNN-LSTM (full)']
    dm = {v: dm_test(full_y - full_p, yy - pp) for v, (yy, pp) in preds.items() if v != 'Proposed CNN-LSTM (full)'}
    return R, dm

ABL = {c: cache(f'ablation_{CC[c]}', lambda c=c: run_ablation(c)) for c in DATA}

## 12. SHAP with a larger explanation set, background resampling, bootstrap CIs and rank stability (R1-6, R2-3, R3-3, R3-10)

In [ ]:
def shap_country(c):
    df = DATA[c]; a, b = SPLITS[c]; out = {}
    for name in DL_MODELS:
        L = LSTAR[c][name]; P = prepare(df, L, a, b)
        hp = BEST_HP[c][name] if c == 'Pakistan' else HP_ROLL[c][name]
        # explained model = the saved hold-out model (seed 0, same configuration); trained and saved only if missing
        mpath = f'{OUT}/models/{CC[c]}_{name}_holdout_seed{SEEDS[0]}.keras'; m = None
        Xexp = np.concatenate([P['va']['X'], P['te']['X']])            # every validation + test window
        Ss = []
        for r in range(SHAP_BG_RESAMPLES):
            spath = f'{OUT}/cache/shap/{CC[c]}_{name}_{TARGET_MODE}_L{L}_n{len(Xexp)}_r{r}.npy'
            if os.path.exists(spath):
                try:
                    Ss.append(np.load(spath)); print(c, name, 'background resample', r, '(loaded from Drive)'); continue
                except Exception:
                    os.remove(spath)
            if m is None:
                if os.path.exists(mpath):
                    m = tf.keras.models.load_model(mpath)
                else:
                    m, _ = fit_dl(name, hp, P, 0); m.save(mpath)
                f = lambda Z, m=m, L=L: m.predict(Z.reshape(-1, L, len(FEATURES)), verbose=0).ravel()
            rng = np.random.default_rng(100 + r)
            bg = P['tr']['X'][rng.choice(len(P['tr']['X']), min(SHAP_BG_SIZE, len(P['tr']['X'])), replace=False)]
            bgs = shap.kmeans(bg.reshape(len(bg), -1), SHAP_BG_KMEANS)
            ex = shap.KernelExplainer(f, bgs)
            sv = np.asarray(ex.shap_values(Xexp.reshape(len(Xexp), -1), nsamples=SHAP_NSAMPLES, silent=True)).reshape(len(Xexp), L, len(FEATURES))
            # rescale to price units (PKR or SAR) so magnitudes are interpretable
            sv = sv * y_unit(P); Ss.append(sv)
            tmp = spath[:-4] + '.tmp.npy'; np.save(tmp, sv); os.replace(tmp, spath)
            print(c, name, 'background resample', r)
        out[name] = dict(S=np.array(Ss), X=Xexp, L=L)
    return out

SHAP = {c: cache(f'shap_{CC[c]}', lambda c=c: shap_country(c)) for c in DATA}

def shap_summary(c):
    rows, taus = [], []
    for name, d in SHAP[c].items():
        S = np.abs(d['S'])                                                # (R, N, L, F)
        grp = S.sum(axis=2).mean(axis=1)                                  # (R, F): mean over windows of |SHAP| summed over lags
        share = 100 * grp / grp.sum(axis=1, keepdims=True)
        # bootstrap over explained windows (first resample) for CIs of the share
        rng = np.random.default_rng(0); N = S.shape[1]; boot = []
        for _ in range(SHAP_BOOT):
            ii = rng.integers(0, N, N); g = S[:, ii].sum(axis=2).mean(axis=1).mean(axis=0); boot.append(100 * g / g.sum())
        boot = np.array(boot)
        for j, f in enumerate(FEATURES):
            rows.append({'Country': c, 'Model': name, 'Feature': f, 'mean_abs_SHAP': grp[:, j].mean(),
                         'share_%': share[:, j].mean(), 'share_lo': np.percentile(boot[:, j], 2.5), 'share_hi': np.percentile(boot[:, j], 97.5),
                         'share_sd_across_backgrounds': share[:, j].std(ddof=1) if len(share) > 1 else np.nan})
        ranks = [stats.rankdata(-g) for g in grp]
        for i, j in itertools.combinations(range(len(ranks)), 2):
            taus.append({'Country': c, 'Model': name, 'tau': stats.kendalltau(ranks[i], ranks[j])[0]})
        lag = S.sum(axis=3).mean(axis=(0, 1))                              # importance per lag
        for k, v in enumerate(lag):
            rows.append({'Country': c, 'Model': name, 'Feature': f'lag t-{d["L"] - 1 - k}', 'mean_abs_SHAP': v})
    return pd.DataFrame(rows), pd.DataFrame(taus)

SH = {c: shap_summary(c) for c in DATA}
SHAP_TAB = pd.concat([SH[c][0] for c in DATA]); SHAP_TAU = pd.concat([SH[c][1] for c in DATA])
print(SHAP_TAB[SHAP_TAB.Feature.isin(FEATURES)].round(2).to_string()); print(SHAP_TAU.groupby(['Country', 'Model']).tau.mean())

## 13. Figures

In [ ]:
def shade_events(ax):
    for k, (a, b) in EVENT_WINDOWS.items():
        ax.axvspan(pd.Timestamp(a), pd.Timestamp(b), alpha=0.08, color='grey')

for c in DATA:
    O = ROLL[c][1]
    fig, ax = plt.subplots(figsize=(12, 5)); shade_events(ax)
    g = O[O.Model == 'Naive']; ax.plot(g.Date, g.y, 'k-', lw=2, label='Actual')
    for mname in ['Naive', 'ARIMAX', 'SVR'] + DL_MODELS:
        g = O[O.Model == mname]; ax.plot(g.Date, g.yhat, lw=1, label=MODEL_LABEL[mname])
    ax.set_title(f'{c}: rolling-origin one-week-ahead forecasts (shaded = event windows)'); ax.set_ylabel(UNITS[c]); ax.legend(ncol=3, fontsize=8)
    fig.tight_layout(); fig.savefig(f'{OUT}/figures/Fig_rolling_{CC[c]}.png', dpi=300); plt.close(fig)
    res = HO[c]; dates = DATA[c].Date.values[res['idx']]
    fig, ax = plt.subplots(figsize=(12, 5)); ax.plot(dates, res['y'], 'k-', lw=2, label='Actual')
    for mname in ['Naive', 'ARIMAX', 'SVR'] + DL_MODELS:
        ax.plot(dates, res['preds'][mname], lw=1, label=MODEL_LABEL[mname])
    ax.set_title(f'{c}: chronological hold-out test period'); ax.set_ylabel(UNITS[c]); ax.legend(ncol=3, fontsize=8)
    fig.tight_layout(); fig.savefig(f'{OUT}/figures/Fig_holdout_{CC[c]}.png', dpi=300); plt.close(fig)
    # SHAP grouped importance with bootstrap CI
    T = SH[c][0]; T = T[T.Feature.isin(FEATURES)]
    fig, ax = plt.subplots(figsize=(10, 4.5)); w = 0.15
    for i, mname in enumerate(DL_MODELS):
        t = T[T.Model == mname].set_index('Feature').loc[FEATURES]
        ax.bar(np.arange(4) + (i - 2) * w, t['share_%'], w, yerr=[np.clip(t['share_%'] - t.share_lo, 0, None), np.clip(t.share_hi - t['share_%'], 0, None)], capsize=2, label=MODEL_LABEL[mname])
    ax.set_xticks(range(4)); ax.set_xticklabels(['Exchange rate', 'Uncertainty index', 'Crude price', 'Lagged petrol price' + (' changes' if TARGET_MODE == 'diff' else '')])
    ax.set_ylabel('Share of total |SHAP| (%)'); ax.set_title(f'{c}: grouped SHAP importance (95% bootstrap CI)'); ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(f'{OUT}/figures/Fig_shap_grouped_{CC[c]}.png', dpi=300); plt.close(fig)
    # SHAP beeswarm for the proposed model (first background resample)
    d = SHAP[c][PROPOSED]; names = [f'{f}_t-{d["L"] - 1 - k}' for k in range(d['L']) for f in FEATURES]
    plt.figure(); shap.summary_plot(d['S'][0].reshape(len(d['X']), -1), d['X'].reshape(len(d['X']), -1), feature_names=names, show=False, max_display=15)
    plt.title(f'{c}: SHAP summary, CNN-LSTM'); plt.tight_layout(); plt.savefig(f'{OUT}/figures/Fig_shap_beeswarm_{CC[c]}.png', dpi=300); plt.close()
    # loss curves of the selected configurations (hold-out, seed 0)
    fig, axes = plt.subplots(1, 5, figsize=(25, 4))
    for ax, mname in zip(axes, DL_MODELS):
        h = HO[c]['hist'][mname]; ax.plot(h['loss'], label='train'); ax.plot(h['val_loss'], label='validation'); ax.set_yscale('log'); ax.set_title(MODEL_LABEL[mname]); ax.legend()
    fig.tight_layout(); fig.savefig(f'{OUT}/figures/Fig_loss_{CC[c]}.png', dpi=200); plt.close(fig)

# data overview (all four weekly series, both countries, event windows shaded)
fig, axes = plt.subplots(4, 2, figsize=(14, 10), sharex=True)
for j, c in enumerate(DATA):
    for i, f in enumerate(FEATURES):
        ax = axes[i, j]; shade_events(ax); ax.plot(DATA[c].Date, DATA[c][f], lw=1)
        ax.set_title(f'{c}: {f.replace("_", " ")}', fontsize=10)
fig.tight_layout(); fig.savefig(f'{OUT}/figures/Fig_data_overview.png', dpi=300); plt.close(fig)

# validation RMSE vs look-back (Pakistan selection)
fig, ax = plt.subplots(figsize=(8, 4.5))
for mname in DL_MODELS:
    g = LB_SUM[LB_SUM.Model == mname]; ax.errorbar(g.L, g['mean'], yerr=g['std'], marker='o', capsize=3, label=MODEL_LABEL[mname])
ax.set_xlabel('Look-back window L (weeks)'); ax.set_ylabel('Validation RMSE (PKR/litre)'); ax.legend(); ax.set_title('Pakistan: look-back selection on the validation block')
fig.tight_layout(); fig.savefig(f'{OUT}/figures/Fig_lookback_PK.png', dpi=300); plt.close(fig)
print('figures written to', f'{OUT}/figures')

## 14. Key numbers for the manuscript placeholders and LaTeX tables

In [ ]:
f2, f3, f4 = '{:.2f}', '{:.3f}', '{:.4f}'
HOS = {c: summarise(HO[c]['R']) for c in DATA}
for c in DATA:
    S = HOS[c].set_index('Model'); cc = CC[c]
    for mname in S.index:
        for k in ['RMSE', 'MAE', 'rRMSE', 'MAPE', 'U2', 'R2', 'MASE']:
            key(f'{cc}_HO_{mname}_{k}', S.loc[mname, k], f4 if c == 'Saudi Arabia' else f3)
            if mname in DL_MODELS:
                key(f'{cc}_HO_{mname}_{k}_SD', S.loc[mname, k + '_sd'], f4)
    dl = S.loc[DL_MODELS].sort_values('RMSE')
    key(f'{cc}_HO_BEST_DL', MODEL_LABEL[dl.index[0]]); key(f'{cc}_HO_BEST_ALL', MODEL_LABEL[S.sort_values('RMSE').index[0]])
    R = ROLL_ALL[ROLL_ALL.Country == c].set_index('Model')
    for mname in R.index:
        for k in ['RMSE', 'MAE', 'rRMSE', 'MAPE', 'U2', 'R2']:
            key(f'{cc}_RO_{mname}_{k}', R.loc[mname, k], f4 if c == 'Saudi Arabia' else f3)
    key(f'{cc}_RO_BEST_DL', MODEL_LABEL[R.loc[DL_MODELS].RMSE.idxmin()]); key(f'{cc}_RO_BEST_ALL', MODEL_LABEL[R.RMSE.idxmin()])
    key(f'{cc}_RO_N_WEEKS', int(R.n_weeks.iloc[0])); key(f'{cc}_RO_N_CHANGES', int(R.n_price_changes.iloc[0]))
    key(f'{cc}_RO_N_FOLDS', len(roll_folds(DATA[c])))
    for mname in DL_MODELS:
        key(f'{cc}_L_{mname}', LSTAR[c][mname])
    dmr = DM_ROLL[(DM_ROLL.Country == c)]
    for _, r in dmr.iterrows():
        key(f'{cc}_DMRO_{r.Model}_vs_{r.vs}_p', r.p, '{:.3f}'); key(f'{cc}_DMRO_{r.Model}_vs_{r.vs}_stat', r.DM, f2)
    dmh = DM_HO[(DM_HO.Country == c)]
    for _, r in dmh.iterrows():
        key(f'{cc}_DMHO_{r.Model}_vs_{r.vs}_p', r.p, '{:.3f}')
    E = EVENTS[(EVENTS.Country == c)]
    for _, r in E.iterrows():
        tagw = ''.join(ch for ch in r.Window.title() if ch.isalnum())
        key(f'{cc}_EV_{tagw}_{r.Model}_rRMSE', r.rRMSE, f3); key(f'{cc}_EV_{tagw}_{r.Model}_U2', r.U2, f3)
    T = SH[c][0]; T = T[T.Feature.isin(FEATURES)]
    for mname in DL_MODELS:
        t = T[T.Model == mname].set_index('Feature')
        key(f'{cc}_SHAP_{mname}_TOP', t['share_%'].idxmax())
        for f in FEATURES:
            key(f'{cc}_SHAP_{mname}_{f}_SHARE', t.loc[f, 'share_%'], '{:.1f}')
            key(f'{cc}_SHAP_{mname}_{f}_CI', f'[{t.loc[f, "share_lo"]:.1f}, {t.loc[f, "share_hi"]:.1f}]')
        key(f'{cc}_SHAP_{mname}_TAU', SH[c][1][SH[c][1].Model == mname].tau.mean(), f2)
    key(f'{cc}_SHAP_N_EXPLAIN', len(SHAP[c][PROPOSED]['X']))
    Rab, dmab = ABL[c]
    g = Rab.groupby('Variant').RMSE.agg(['mean', 'std'])
    for v in g.index:
        vk = ''.join(ch for ch in v.title() if ch.isalnum())
        key(f'{cc}_ABL_{vk}_RMSE', g.loc[v, 'mean'], f4 if c == 'Saudi Arabia' else f3)
        if v in dmab: key(f'{cc}_ABL_{vk}_DMp', dmab[v][1], '{:.3f}')
key('ARIMA_ORDER_PK', str(HO['Pakistan']['cfg']['ARIMA'])); key('ARIMAX_ORDER_PK', str(HO['Pakistan']['cfg']['ARIMAX']))
key('ARIMA_ORDER_SA', str(HO['Saudi Arabia']['cfg']['ARIMA'])); key('ARIMAX_ORDER_SA', str(HO['Saudi Arabia']['cfg']['ARIMAX']))
key('SVR_CFG_PK', str(HO['Pakistan']['cfg']['SVR'])); key('SVR_CFG_SA', str(HO['Saudi Arabia']['cfg']['SVR']))
for k, v in ENV.items(): key(f'ENV_{k}', str(v))
key('N_TRIALS', N_TRIALS); key('N_SEEDS', len(SEEDS)); key('TARGET_MODE', TARGET_MODE); key('CRUDE_LAG_FIX', CRUDE_LAG_FIX)

pd.DataFrame([{'ID': k, 'Value': v} for k, v in KEY.items()]).to_csv(f'{OUT}/R2_key_numbers.csv', index=False)
print(len(KEY), 'placeholder values written to', f'{OUT}/R2_key_numbers.csv')

# ---------- LaTeX tables (paste into the revised manuscript in place of the blue placeholder tables) ----------
def fmt(v, d=3):
    return '--' if v is None or (isinstance(v, float) and np.isnan(v)) else f'{v:.{d}f}'

def tex_table(rows, header, caption, label, colspec):
    # Emits ONLY the tabular block: the caption and label already sit in the revised manuscript (caption/label kept here as a comment).
    L_ = [f'% {label}: {caption}', '\\resizebox{\\linewidth}{!}{%',
          f'\\begin{{tabular}}{{{colspec}}}', '\\hline', ' & '.join(header) + ' \\\\ \\hline']
    L_ += [' & '.join(map(str, r)) + ' \\\\' for r in rows]
    L_ += ['\\hline', '\\end{tabular}}']
    return '\n'.join(L_)

ORDER = ['Naive', 'SeasonalNaive', 'RWDrift', 'ARIMA', 'ARIMAX', 'SVR'] + DL_MODELS
for c in DATA:
    d = 4 if c == 'Saudi Arabia' else 3; S = HOS[c].set_index('Model'); rows = []
    for mname in ORDER:
        r = S.loc[mname]; pm = (lambda k: f'{fmt(r[k], d)} $\\pm$ {fmt(r[k + "_sd"], d)}') if mname in DL_MODELS else (lambda k: fmt(r[k], d))
        rows.append([MODEL_LABEL[mname], pm('RMSE'), pm('MAE'), fmt(r.rRMSE, 2), fmt(r.MAPE, 2), fmt(r.MASE, 3), fmt(r.U2, 3), fmt(r.R2, 3)])
    open(f'{OUT}/tables/tab_holdout_{CC[c]}.tex', 'w').write(tex_table(rows, ['Model', f'RMSE ({UNITS[c]})', 'MAE', 'rRMSE (\\%)', 'MAPE (\\%)', 'MASE', "Theil's U2", '$R^2$'],
        f'Chronological hold-out test results, {c}.', f'tab:holdout_{CC[c]}', 'l' + 'c' * 7))
rows = []
for mname in ORDER:
    r1 = ROLL_ALL[(ROLL_ALL.Country == 'Pakistan') & (ROLL_ALL.Model == mname)].iloc[0]; r2 = ROLL_ALL[(ROLL_ALL.Country == 'Saudi Arabia') & (ROLL_ALL.Model == mname)].iloc[0]
    rows.append([MODEL_LABEL[mname], fmt(r1.RMSE), fmt(r1.rRMSE, 2), fmt(r1.U2), fmt(r2.RMSE, 4), fmt(r2.rRMSE, 2), fmt(r2.U2)])
open(f'{OUT}/tables/tab_rolling.tex', 'w').write(tex_table(rows, ['Model', 'PK RMSE', 'PK rRMSE (\\%)', 'PK U2', 'SA RMSE', 'SA rRMSE (\\%)', 'SA U2'],
    'Rolling-origin out-of-sample results, 2019--2026.', 'tab:rolling', 'lcccccc'))
for c in DATA:
    E = EVENTS[(EVENTS.Country == c) & (EVENTS.Split == 'event')]; rows = []
    for w in list(EVENT_WINDOWS):
        e = E[E.Window == w]
        if e.empty: continue
        g = lambda m: e[e.Model == m].iloc[0] if (e.Model == m).any() else None
        rows.append([w, int(e.n_weeks.iloc[0]), int(e.n_price_changes.iloc[0])] +
                    [fmt(g(m).rRMSE, 2) if g(m) is not None else '--' for m in ['Naive', 'ARIMAX', 'SVR', 'CNNGRU', 'CNNLSTM']] +
                    [fmt(g('CNNLSTM').U2, 3) if g('CNNLSTM') is not None else '--'])
    open(f'{OUT}/tables/tab_events_{CC[c]}.tex', 'w').write(tex_table(rows, ['Window', 'Weeks', 'Changes', 'Naive', 'ARIMAX', 'SVR', 'CNN-GRU', 'CNN-LSTM', 'CNN-LSTM U2'],
        f'Event-window rRMSE (\\%) of rolling-origin forecasts, {c}.', f'tab:events_{CC[c]}', 'lcccccccc'))
rows = []
for c in DATA:
    for _, r in VIF[VIF.Country == c].iterrows():
        rows.append([c, r.Variable.replace('_', ' '), fmt(r['VIF (levels)'], 2), fmt(r['VIF (first differences)'], 2)])
open(f'{OUT}/tables/tab_vif.tex', 'w').write(tex_table(rows, ['Country', 'Variable', 'VIF (levels)', 'VIF (first differences)'], 'Variance inflation factors.', 'tab:vif', 'llcc'))
rows = []
for c in DATA:
    Rab, dmab = ABL[c]; g = Rab.groupby('Variant').RMSE.agg(['mean', 'std'])
    for v in ABL_VARIANTS:
        rows.append([c, v, fmt(g.loc[v, 'mean'], 4 if c == 'Saudi Arabia' else 3), fmt(g.loc[v, 'std'], 4), fmt(dmab[v][1], 3) if v in dmab else '--'])
open(f'{OUT}/tables/tab_ablation.tex', 'w').write(tex_table(rows, ['Country', 'Variant', 'RMSE (mean)', 'SD', 'DM p vs full'], 'Ablation study.', 'tab:ablation', 'llccc'))
rows = []
for c in DATA:
    T = SH[c][0]; T = T[T.Feature.isin(FEATURES)]
    for mname in DL_MODELS:
        t = T[T.Model == mname].set_index('Feature')
        rows.append([c, MODEL_LABEL[mname]] + [f'{t.loc[f, "share_%"]:.1f} [{t.loc[f, "share_lo"]:.1f}, {t.loc[f, "share_hi"]:.1f}]' for f in FEATURES] +
                    [fmt(SH[c][1][SH[c][1].Model == mname].tau.mean(), 2)])
open(f'{OUT}/tables/tab_shap.tex', 'w').write(tex_table(rows, ['Country', 'Model', 'Exchange rate', 'Uncertainty', 'Crude price', 'Lagged price', "Kendall's $\\tau$"],
    'Grouped SHAP importance (share of total |SHAP|, \\%, 95\\% bootstrap CI).', 'tab:shap', 'llccccc'))
rows = []
for m in DL_MODELS:
    hp = BEST_HP['Pakistan'][m]
    rows.append([MODEL_LABEL[m], LSTAR['Pakistan'][m], LSTAR['Saudi Arabia'][m]] + [hp.get(k, '--') for k in ['n_conv', 'filters', 'kernel', 'n_rec', 'units', 'dense', 'dropout', 'lr', 'batch', 'loss']])
open(f'{OUT}/tables/tab_selected_config.tex', 'w').write(tex_table(rows, ['Model', '$L^*$ PK', '$L^*$ SA', 'Conv layers', 'Filters', 'Kernel', 'Rec. layers', 'Units', 'Dense', 'Dropout', 'LR', 'Batch', 'Loss'],
    'Selected look-back windows (weeks) and hyper-parameters.', 'tab:selected', 'l' + 'c' * 12))
rows = [[int(r.n_conv), int(r.n_rec), fmt(r['min'], 3), fmt(r['median'], 3), int(r['count'])] for _, r in SENS_DEPTH.iterrows()]
open(f'{OUT}/tables/tab_cnnlstm_depth.tex', 'w').write(tex_table(rows, ['Conv layers', 'LSTM layers', 'Best val. RMSE', 'Median val. RMSE', 'Configs'],
    'Depth sensitivity of CNN-LSTM (Pakistan validation block).', 'tab:depth', 'ccccc'))
rows = []
for c in DATA:
    for f in ['Exchange_Rate', 'Uncertainty_Index', 'Global_Oil_Price']:
        g = GC[(GC.Country == c) & (GC.Cause == f)]
        rows.append([c, f.replace('_', ' ')] + [fmt(p, 3) for p in g.p.values])
open(f'{OUT}/tables/tab_granger.tex', 'w').write(tex_table(rows, ['Country', 'Predictor', 'p (lag 1)', 'p (lag 2)', 'p (lag 3)', 'p (lag 4)'],
    'Granger-causality tests (first differences) of each input for the domestic petrol price.', 'tab:granger', 'llcccc'))

with pd.ExcelWriter(f'{OUT}/Petrol_Price_Results_R2.xlsx') as xw:
    DESC.to_excel(xw, sheet_name='Data_description', index=False); VIF.to_excel(xw, sheet_name='VIF', index=False); ADF.to_excel(xw, sheet_name='ADF', index=False); GC.to_excel(xw, sheet_name='Granger', index=False)
    LB.to_excel(xw, sheet_name='Lookback_selection_PK', index=False); HS.to_excel(xw, sheet_name='Random_search_PK', index=False)
    for c in DATA:
        HO[c]['R'].to_excel(xw, sheet_name=f'Holdout_seeds_{CC[c]}', index=False); HOS[c].to_excel(xw, sheet_name=f'Holdout_summary_{CC[c]}', index=False)
        ROLL[c][0].to_excel(xw, sheet_name=f'Rolling_folds_{CC[c]}', index=False); ROLL[c][1].to_excel(xw, sheet_name=f'Rolling_OOS_{CC[c]}', index=False)
        ABL[c][0].to_excel(xw, sheet_name=f'Ablation_{CC[c]}', index=False)
    ROLL_ALL.to_excel(xw, sheet_name='Rolling_summary', index=False); EVENTS.to_excel(xw, sheet_name='Event_windows', index=False)
    DM_HO.to_excel(xw, sheet_name='DM_holdout', index=False); DM_ROLL.to_excel(xw, sheet_name='DM_rolling', index=False)
    SHAP_TAB.to_excel(xw, sheet_name='SHAP', index=False); SHAP_TAU.to_excel(xw, sheet_name='SHAP_rank_stability', index=False)
    pd.DataFrame([{'ID': k, 'Value': v} for k, v in KEY.items()]).to_excel(xw, sheet_name='Key_numbers', index=False)
print('Excel and LaTeX tables written.')

## 15. Grad-CAM saliency for the convolutional models (retained from v5, now on the chronological test windows)

In [ ]:
def gradcam_1d(model, x, conv_name='conv0'):
    """1-D Grad-CAM over the time (lag) axis of the first convolutional layer.
    Always returns a 1-D array of length T (time steps of the conv output), even when T == 1."""
    gm = Model(model.inputs, [model.get_layer(conv_name).output, model.output])
    with tf.GradientTape() as tape:
        co, pr = gm(x)
        loss = pr[:, 0]
    g = tape.gradient(loss, co)                      # (1, T, filters)
    w = tf.reduce_mean(g, axis=(0, 1))               # (filters,)
    h = tf.maximum(tf.reshape(co[0] @ w[:, None], [-1]), 0.0)   # reshape, not squeeze: keeps T == 1
    mx = tf.reduce_max(h)
    return (h / mx).numpy() if mx > 0 else h.numpy()

for c in DATA:
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
    for ax, mname in zip(axes, ['CNNLSTM', 'CNNLSTMAttention', 'CNNGRU']):
        p = f'{OUT}/models/{CC[c]}_{mname}_holdout_seed{SEEDS[0]}.keras'
        if not os.path.exists(p):
            ax.set_axis_off(); continue
        m = tf.keras.models.load_model(p)
        a, b = SPLITS[c]; P = prepare(DATA[c], LSTAR[c][mname], a, b); Xte = P['te']['X']
        H = np.array([gradcam_1d(m, Xte[i:i + 1]) for i in range(len(Xte))]).reshape(len(Xte), -1)
        im = ax.imshow(H.T, aspect='auto', cmap='viridis', vmin=0, vmax=1, interpolation='nearest', origin='lower')
        ax.set_title(f'{MODEL_LABEL[mname]} (L* = {LSTAR[c][mname]} wk, {H.shape[1]} conv step(s))')
        ax.set_xlabel('test week'); ax.set_ylabel('conv time step (oldest → newest)'); ax.set_yticks(range(H.shape[1]))
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle(f'{c}: Grad-CAM over the hold-out test windows'); fig.tight_layout()
    fig.savefig(f'{OUT}/figures/Fig_gradcam_{CC[c]}.png', dpi=200); plt.close(fig)
print('Grad-CAM figures written')


## 15b. Sensitivity: R1 level target vs R2 change target (same architecture, same settings)
The five networks are re-trained with the R1 min–max **level** representation, using the R2-selected look-back and hyper-parameters, on the hold-out (both countries, 5 seeds) and the rolling-origin protocol (both countries, L fixed at L*, 3 seeds). The table reports both, so the effect of the representation is shown openly.


In [ ]:
def level_holdout(c):
    df = DATA[c]; a, b = SPLITS[c]; rows = []
    hp_map = BEST_HP['Pakistan'] if c == 'Pakistan' else HP_ROLL[c]
    for name in DL_MODELS:
        P = prepare(df, LSTAR[c][name], a, b, mode='level')
        for s in SEEDS:
            p = fit_eval(name, hp_map[name], P, s)['pred']['te']
            rows.append({'Country': c, 'Model': name, 'seed': s, **metrics(P['te']['y'], p, P['te']['last'])})
    return pd.DataFrame(rows)

def level_rolling(c):
    df = DATA[c]; y = df[TARGET].values; oos = []
    hp_map = BEST_HP['Pakistan'] if c == 'Pakistan' else HP_ROLL[c]
    for k, (s, e) in enumerate(roll_folds(df)):
        tr_end = int(s - ROLL_VAL_FRAC * s)
        for name in DL_MODELS:
            P = prepare(df, LSTAR[c][name], tr_end, s, e, mode='level')
            p = np.mean([fit_eval(name, hp_map[name], P, sd)['pred']['te'] for sd in ROLL_SEEDS], axis=0)
            for t, v in zip(P['te']['t'], p):
                oos.append({'Country': c, 'Model': name, 't': int(t), 'y': y[t], 'yhat': float(v), 'y_naive': y[t - HORIZON]})
        print(c, 'level comparison fold', k)
    return pd.DataFrame(oos)

if RUN_LEVEL_COMPARISON:
    LVL_HO = {c: cache(f'level_holdout_{CC[c]}', lambda c=c: level_holdout(c)) for c in DATA}
    LVL_RO = {c: cache(f'level_rolling_{CC[c]}', lambda c=c: level_rolling(c)) for c in DATA}
    rows = []
    for c in DATA:
        hoL = summarise(LVL_HO[c], metric_cols=('RMSE', 'rRMSE', 'U2')).set_index('Model'); hoD = HOS[c].set_index('Model')
        roL = oos_metrics(LVL_RO[c]).set_index('Model'); roD = ROLL_ALL[ROLL_ALL.Country == c].set_index('Model')
        for m in DL_MODELS:
            rows.append({'Country': c, 'Model': MODEL_LABEL[m],
                         'HO U2 level': hoL.loc[m, 'U2'], 'HO U2 change': hoD.loc[m, 'U2'],
                         'RO U2 level': roL.loc[m, 'U2'], 'RO U2 change': roD.loc[m, 'U2']})
            key(f'{CC[c]}_CMP_{m}_HO_U2_level', hoL.loc[m, 'U2'], '{:.3f}'); key(f'{CC[c]}_CMP_{m}_RO_U2_level', roL.loc[m, 'U2'], '{:.3f}')
    CMP = pd.DataFrame(rows); print(CMP.round(3).to_string())
    open(f'{OUT}/tables/tab_target_comparison.tex', 'w').write(tex_table(
        [[r.Country, r.Model, fmt(r['HO U2 level']), fmt(r['HO U2 change']), fmt(r['RO U2 level']), fmt(r['RO U2 change'])] for _, r in CMP.iterrows()],
        ['Country', 'Model', 'Hold-out U2 (level)', 'Hold-out U2 (change)', 'Rolling U2 (level)', 'Rolling U2 (change)'],
        "Theil's U2 of the networks with the level (R1) and change (R2) representations.", 'tab:target_cmp', 'llcccc'))
    pd.DataFrame([{'ID': k, 'Value': v} for k, v in KEY.items()]).to_csv(f'{OUT}/R2_key_numbers.csv', index=False)
    with pd.ExcelWriter(f'{OUT}/Petrol_Price_Results_R2.xlsx', mode='a', engine='openpyxl', if_sheet_exists='replace') as xw:
        CMP.to_excel(xw, sheet_name='Level_vs_change', index=False)
        pd.DataFrame([{'ID': k, 'Value': v} for k, v in KEY.items()]).to_excel(xw, sheet_name='Key_numbers', index=False)
    print('Level-vs-change comparison written.')


## 16. Package everything for the public repository (Editor-4, R1-15, R2-8, R4)

In [ ]:
import zipfile
PKG = os.path.join(DRIVE_ROOT, os.path.basename(OUT) + '_package.zip')
with zipfile.ZipFile(PKG, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files in os.walk(OUT):
        rel_root = os.path.relpath(root, OUT)
        if rel_root.split(os.sep)[0] == 'cache':      # checkpoints are not part of the public package
            continue
        for fn in files:
            if '.tmp' in fn: continue
            z.write(os.path.join(root, fn), os.path.join(os.path.basename(OUT), rel_root, fn))
print('Package saved to:', PKG)
print('Upload the package + this notebook + the two raw CSV files to Zenodo/Figshare/OSF and paste the DOI into the manuscript.')
